# Physical Climate Risk: Tropical Cyclone Trends Affecting Western Australia (1985–2024)

**A repeatable analysis written for AASB S2 physical-risk assessment.** (AASB S2 is the Australian rule that asks big companies to report their climate risks.)

**The question I asked.** Have the tropical cyclones that hit Western Australia grown stronger between 1985 and 2024? And do they track the warming of the sea?

**What I found.** Over this period the seas off WA warmed by about 0.5 °C. That warming is a solid, real trend. Even so, the cyclones that affect WA did not get stronger. Their average strength drifted slightly lower, and it showed no positive link with sea-surface temperature (how warm the top layer of the ocean is). The lesson for climate-risk reporting is simple: you cannot guess future cyclone danger from the recent local record. You need forward-looking projections instead.

Data: IBTrACS v04r01 (NOAA NCEI), NOAA ERSSTv5, and the Bureau of Meteorology Southern-Hemisphere database. The statistics are coded from scratch in `stats_utils.py` and checked in `test_stats.py`.

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from IPython.display import Image, display
import analysis  # the four analysis steps, as functions

have_raw = Path('data/raw/ibtracs.SI.list.v04r01.csv').exists()
print('Raw source data present:', have_raw,
      '\n(If False, the notebook runs from the committed cleaned CSVs.)')

## 1. Data and cleaning

I load IBTrACS, keep the seasons from 1985–2024, and boil the track points down to one row per storm: peak BOM 10-minute wind, lowest central pressure, peak US 1-minute wind, and how close the storm came to the WA coast. A storm counts as *WA-affecting* if its track passes within 500 km of the coast between the north Kimberley and the Mid West.

**Why I use the BOM 10-minute wind.** Different agencies average wind speed over different lengths of time (the US uses 1 minute, BOM uses 10 minutes), so their wind numbers are not the same thing. The BOM value comes out about 12% lower. I lead with the BOM figure because the subject is WA, and I check it against US winds and central pressure.

**Checking the data.** The BOM winds match the Bureau's own database to the knot for the major storms. The 12% US/BOM gap is exactly what the averaging difference predicts. And 97% of the flagged WA storms also show up in the Bureau's Australian database. Within the WA group, the data has a BOM wind figure 92% of the time (85% in the 1980s, rising to 100% in recent years).

In [ ]:
storms = analysis.load_storms()
wa = storms[storms.wa_affecting].copy()
print(f'{len(storms)} South Indian Ocean storms, {len(wa)} WA-affecting, '
      f'seasons {storms.season.min()}-{storms.season.max()}')
display(wa[['season','name','peak_bom_wind_kt','min_bom_pres_hpa',
            'peak_usa_wind_kt','peak_usa_sshs','min_dist_wa_km']].head(10))

## 2. Frequency and intensity by decade

In a typical season, about five cyclones come within 500 km of WA. Their average peak strength has drifted **down** over the four decades, not up: mean BOM wind fell from about 76 kt to 63 kt, and mean pressure rose (a higher pressure means a weaker storm).

In [ ]:
summary, all_count, wa_count = analysis.explore(storms)
display(summary.round(1))
display(Image('charts/01_annual_count.png'))
display(Image('charts/02_intensity_by_decade.png'))

## 3. Trend tests

I test each year-by-year series three ways: ordinary least squares (OLS, the standard straight-line fit) for the slope, Mann-Kendall for whether the trend is real (this is the standard test in climate science), and Sen's slope for a robust estimate of how big the trend is.

The direction always points toward weaker storms. For the WA-only group the trend is not statistically solid, because the sample is small (only about five storms a year). Across the whole South Indian Ocean, where there are far more storms, the drop in mean wind is statistically significant. Two separate measures, wind and pressure, point the same way.

In [ ]:
trend_table = analysis.trends(storms)
display(trend_table.round(3))
display(Image('charts/03_trend_wind_speed.png'))
display(Image('charts/04_trend_pressure.png'))

## 4. Rapid intensification

Rapid intensification means a storm gaining at least 30 kt of wind in 24 hours. The share of storms that did this (measured on US 1-minute winds, the usual convention for this metric) rose from about 21% in 1985–94 to around 40% since.

**A caveat I want to be clear about.** Older best-track data is smoother and recorded less often, which by itself makes rapid intensification harder to spot in the early years. So part of the apparent rise is probably an artefact of how storms were observed back then, not a purely physical change. Treat it as suggestive, not proven.

In [ ]:
ri_table = analysis.rapid_intensification(storms)
display(ri_table.round(1))
display(Image('charts/05_rapid_intensification.png'))

## 5. Sea-surface temperature

From ERSSTv5 I take the WA cyclone development region (10–30°S, 70–130°E), work out monthly anomalies (how far each month sits above or below normal) against the 1991–2020 average, and take the mean over the Nov–Apr cyclone season. I then check whether that seasonal anomaly lines up with the yearly average strength of WA-affecting storms.

The ocean warmed by a clear, significant amount (about 0.16 °C per decade). Yet warmer seasons were **not** linked to stronger storms. This split is the core of the project: warm water sets how much energy is on the table, but wind shear (winds changing speed or direction with height, which can tear a storm apart) and the big-picture circulation (ENSO, the Indian Ocean Dipole) decide whether that energy is actually used.

In [ ]:
sst = analysis.sst_correlation(storms)
print(f"SST trend: {sst['sst_trend_decade']:+.3f} degC/decade (p={sst['sst_p']:.1e})")
print(f"SST vs WA wind:     r={sst['r_wind']:+.2f}, p={sst['p_wind']:.3f}")
print(f"SST vs WA pressure: r={sst['r_pres']:+.2f}, p={sst['p_pres']:.3f}")
display(Image('charts/06_sst_correlation.png'))

## Findings and AASB S2 implications

Here is the picture. Frequency is flat to slightly down. Strength has not risen, and if anything it has eased (clearly so across the basin, only marginally for WA alone). Rapid intensification looks more common, with the data caveat above. And although the ocean warmed clearly, that warming did not turn into stronger WA cyclones in the record we have.

For AASB S2 physical-risk reporting, which starts for large WA entities from 1 July 2026, the message is this: the historical record does not back a simple 'warmer oceans mean stronger storms' claim for WA. But the lack of an observed trend is not proof of safety. The energy ceiling has risen, and projections still warn of fewer but possibly more intense systems. So good reporting should rest on forward-looking scenarios, not on assuming the past will repeat.

See `README.md` for the full write-up and `INTERVIEW_BRIEF.md` for talking points.